####
Lemmatization (using spaCy) reduces a word to its base or dictionary form by removing inflectional endings.
Example: workers → worker (plural → singular)
Example: running → run (tense normalization)
However, lemmatization does not handle derivational morphology.
Example: worker ≠ work
Here, “er” is a derivational suffix that changes the word class (verb → noun).
Words can contain multiple morphemes:
workers = work + er + s
“er” → derivational morpheme
“s” → inflectional morpheme

In [15]:


# Inflectional suffixes (do NOT change word meaning, only grammar)
inflectional_suffixes = ['s', 'es', 'ed', 'ing']
# Derivational suffixes (change meaning or part of speech)
derivational_suffixes = ['ness', 'ly', 'ful', 'able', 'ment', 'ize', 'al', 'ous', 'ship', 'tion', 'er']

In [2]:
def classify_word(word):
    for suffix in derivational_suffixes:
        if word.endswith(suffix):
            return "Derivational"
    
    for suffix in inflectional_suffixes:
        if word.endswith(suffix):
            return "Inflectional"
    
    return "Root/Unknown"

In [3]:
words = ["playing", "happiness", "dogs", "quickly", "teacher", "books", "national"]

for word in words:
    print(word, "->", classify_word(word))

playing -> Inflectional
happiness -> Derivational
dogs -> Inflectional
quickly -> Derivational
teacher -> Inflectional
books -> Inflectional
national -> Derivational


In [4]:
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /home/amisha/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/amisha/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

print(lemmatizer.lemmatize("better", pos='a')) 

good


In [6]:

def identify_morpheme_type(word):
    lemmatizer = WordNetLemmatizer()
    
    # Tokenize the word
    tokens = word_tokenize(word)
    
    for token in tokens:
        # Check for inflectional morphemes
        for suffix in inflectional_suffixes:
            if token.endswith(suffix):
                # Lemmatize the word 
                lemma = lemmatizer.lemmatize(token, pos='v')  # Lemmatize as a verb for tense forms
                return f"Inflectional Morpheme: {suffix} (Word: {token}, Lemma: {lemma})"
        
        # Check for derivational morphemes
        for suffix in derivational_suffixes:
            if token.endswith(suffix):
                return f"Derivational Morpheme: {suffix} (Word: {token})"
    
    return "No Morpheme Found"


test_words = ['running', 'happiness', 'cats', 'quickly', 'buildable', 'worker', 'nation']
for word in test_words:
    print(f"Word: {word} -> {identify_morpheme_type(word)}")


Word: running -> Inflectional Morpheme: ing (Word: running, Lemma: run)
Word: happiness -> Inflectional Morpheme: s (Word: happiness, Lemma: happiness)
Word: cats -> Inflectional Morpheme: s (Word: cats, Lemma: cat)
Word: quickly -> Derivational Morpheme: ly (Word: quickly)
Word: buildable -> Derivational Morpheme: able (Word: buildable)
Word: worker -> Inflectional Morpheme: er (Word: worker, Lemma: worker)
Word: nation -> Derivational Morpheme: tion (Word: nation)


In [7]:
nltk.download('omw-1.4')

[nltk_data] Downloading package omw-1.4 to /home/amisha/nltk_data...


True

In [9]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [22]:
def classify_morpheme(text):
    doc = nlp(text)
    results = []

    for token in doc:
        word = token.text.lower()
        lemma = token.lemma_
        pos = token.pos_

        # 🔹 Skip auxiliary verbs (like is, are, was)
        if pos == "AUX":
            results.append(f"{word} -> Root/Irregular")
            continue

        # 🔹 Step 1: Derivational FIRST
        for suffix in sorted(derivational_suffixes, key=len, reverse=True):
            if word.endswith(suffix):
                results.append(f"{word} -> Derivational (Suffix: {suffix})")
                break
        else:
            # 🔹 Step 2: Inflectional (only if suffix-based change)
            if word != lemma and any(word.endswith(s) for s in inflectional_suffixes):
                results.append(f"{word} -> Inflectional (Lemma: {lemma})")
            else:
                results.append(f"{word} -> Root/Unknown")

    return results

In [23]:
text = "The workers are running quickly and building happiness in nations"

output = classify_morpheme(text)

for line in output:
    print(line)

the -> Root/Unknown
workers -> Inflectional (Lemma: worker)
are -> Root/Irregular
running -> Inflectional (Lemma: run)
quickly -> Derivational (Suffix: ly)
and -> Root/Unknown
building -> Inflectional (Lemma: build)
happiness -> Derivational (Suffix: ness)
in -> Root/Unknown
nations -> Inflectional (Lemma: nation)
